# 09 — XGBoost y LightGBM con Optuna

Tercer y cuarto modelo del proyecto. Objetivos:
1. Entrenar XGBoost con búsqueda Optuna (50 trials).
2. Entrenar LightGBM con búsqueda Optuna (100 trials).
3. Optimizar `recall_critico` (> 0.85) como métrica primaria.
4. Registrar experimentos en MLflow (DagsHub), versionar modelos con DVC.
5. Comparar contra modelos anteriores (ver notebook 10).

## Métricas del proyecto
| Métrica | Umbral |
|---------|--------|
| `recall_critico` | > 0.85 |
| `precision_critico` | > 0.70 |
| `f1_macro` | > 0.65 |

In [ ]:
import os
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, learning_curve
from xgboost import XGBClassifier

from sonalert.config import FIGURES_DIR, MODELS_DIR, PROCESSED_DATA_DIR, PROJ_ROOT

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
CLASS_NAMES = ["bajo", "medio", "alto", "critico"]
CRITICO = 3
N_TRIALS_XGB = 50
N_TRIALS_LGBM = 100

In [ ]:
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI")
MLFLOW_ACTIVE = bool(MLFLOW_URI)

if MLFLOW_ACTIVE:
    import mlflow
    import mlflow.sklearn

    mlflow.set_tracking_uri(MLFLOW_URI)
    mlflow.set_experiment("xgb_lgbm_optuna")
    print(f"[MLflow] activo: {MLFLOW_URI}")
else:
    print("[MLflow] inactivo — figuras y modelos se guardan localmente.")

## 2. Cargar splits preprocesados

In [ ]:
X_train   = pd.read_parquet(PROCESSED_DATA_DIR / "X_train.parquet")
X_val     = pd.read_parquet(PROCESSED_DATA_DIR / "X_val.parquet")
X_holdout = pd.read_parquet(PROCESSED_DATA_DIR / "X_holdout.parquet")

y_train   = pd.read_parquet(PROCESSED_DATA_DIR / "y_train.parquet")["y"]
y_val     = pd.read_parquet(PROCESSED_DATA_DIR / "y_val.parquet")["y"]
y_holdout = pd.read_parquet(PROCESSED_DATA_DIR / "y_holdout.parquet")["y"]

print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}   | y_val:   {y_val.shape}")
print(f"X_holdout: {X_holdout.shape} | y_holdout: {y_holdout.shape}")
print(f"\nClass balance train: {dict(y_train.value_counts(normalize=True).sort_index().round(3))}")

## 3. Scorers y función de evaluación

In [ ]:
scorers = {
    "recall_critico":    make_scorer(recall_score, labels=[CRITICO], average="macro", zero_division=0),
    "precision_critico": make_scorer(precision_score, labels=[CRITICO], average="macro", zero_division=0),
    "f1_macro":          make_scorer(f1_score, average="macro", zero_division=0),
    "accuracy":          "accuracy",
}
PRIMARY_METRIC = "recall_critico"
tscv = TimeSeriesSplit(n_splits=5)


def evaluate(model, X, y, split_name: str) -> tuple:
    y_pred = model.predict(X)
    metrics = {
        f"{split_name}_recall_critico":    recall_score(y, y_pred, labels=[CRITICO], average="macro", zero_division=0),
        f"{split_name}_precision_critico": precision_score(y, y_pred, labels=[CRITICO], average="macro", zero_division=0),
        f"{split_name}_f1_macro":          f1_score(y, y_pred, average="macro", zero_division=0),
        f"{split_name}_accuracy":          float((y_pred == y).mean()),
    }
    print(f"\n=== {split_name.upper()} ===")
    print(classification_report(y, y_pred, target_names=CLASS_NAMES, digits=3, zero_division=0))
    for k, v in metrics.items():
        print(f"  {k}: {v:.3f}")
    return metrics, y_pred


print(f"Métrica primaria: {PRIMARY_METRIC}")

## 4. XGBoost con Optuna

**Espacio de búsqueda:**
- `n_estimators`: int [50, 500]
- `max_depth`: int [3, 10]
- `learning_rate`: log-uniform [1e-3, 0.3]
- `subsample`: uniform [0.5, 1.0]
- `colsample_bytree`: uniform [0.5, 1.0]
- `reg_alpha`: log-uniform [1e-4, 10]
- `reg_lambda`: log-uniform [1e-4, 10]
- `min_child_weight`: int [1, 10]

In [ ]:
def xgb_objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 50, 500),
        "max_depth":         trial.suggest_int("max_depth", 3, 10),
        "learning_rate":     trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "min_child_weight":  trial.suggest_int("min_child_weight", 1, 10),
        "objective":         "multi:softprob",
        "num_class":         4,
        "eval_metric":       "mlogloss",
        "verbosity":         0,
        "n_jobs":            -1,
        "random_state":      RANDOM_STATE,
    }
    model = XGBClassifier(**params)
    scores = cross_val_score(
        model, X_train, y_train,
        cv=tscv,
        scoring=scorers[PRIMARY_METRIC],
        n_jobs=1,
    )
    return scores.mean()

In [ ]:
xgb_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    study_name="xgb_optuna",
)
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS_XGB, show_progress_bar=True)

print(f"\nMejor trial XGB:")
print(f"  CV {PRIMARY_METRIC}: {xgb_study.best_value:.4f}")
print(f"  Params: {xgb_study.best_params}")

In [ ]:
best_xgb_params = xgb_study.best_params.copy()
best_xgb_params.update({
    "objective":    "multi:softprob",
    "num_class":    4,
    "eval_metric":  "mlogloss",
    "verbosity":    0,
    "n_jobs":       -1,
    "random_state": RANDOM_STATE,
})

xgb_best = XGBClassifier(**best_xgb_params)
xgb_best.fit(X_train, y_train)

xgb_metrics_val,     y_val_pred_xgb     = evaluate(xgb_best, X_val,     y_val,     "val")
xgb_metrics_holdout, y_holdout_pred_xgb = evaluate(xgb_best, X_holdout, y_holdout, "holdout")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (y_true, y_pred, title) in zip(
    axes,
    [(y_val, y_val_pred_xgb, "VAL"), (y_holdout, y_holdout_pred_xgb, "HOLDOUT")],
):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d"
    )
    ax.set_title(f"XGB best — {title}")
plt.tight_layout()
xgb_cm_path = FIGURES_DIR / "09_xgb_confusion.png"
plt.savefig(xgb_cm_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Guardado: {xgb_cm_path.relative_to(PROJ_ROOT)}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
xgb_trials_df = xgb_study.trials_dataframe()
ax.plot(xgb_trials_df.index, xgb_trials_df["value"], alpha=0.5, marker=".", color="#2E86AB")
ax.plot(xgb_trials_df.index, xgb_trials_df["value"].cummax(), color="#E63946", linewidth=2, label="best so far")
ax.set_xlabel("Trial")
ax.set_ylabel(PRIMARY_METRIC)
ax.set_title("XGB — historial Optuna")
ax.legend()
ax.grid(alpha=0.3)
xgb_hist_path = FIGURES_DIR / "09_xgb_optuna_history.png"
plt.savefig(xgb_hist_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Guardado: {xgb_hist_path.relative_to(PROJ_ROOT)}")

In [ ]:
xgb_model_path = MODELS_DIR / "xgb_best.joblib"
joblib.dump(xgb_best, xgb_model_path)
print(f"Modelo XGB guardado: {xgb_model_path.relative_to(PROJ_ROOT)}")

## 5. LightGBM con Optuna

**Espacio de búsqueda:**
- `n_estimators`: int [50, 500]
- `num_leaves`: int [20, 150]
- `learning_rate`: log-uniform [1e-3, 0.3]
- `min_child_samples`: int [5, 100]
- `subsample`: uniform [0.5, 1.0]
- `colsample_bytree`: uniform [0.5, 1.0]
- `reg_alpha`: log-uniform [1e-4, 10]
- `reg_lambda`: log-uniform [1e-4, 10]
- `class_weight`: balanced / None

In [ ]:
def lgbm_objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 50, 500),
        "num_leaves":        trial.suggest_int("num_leaves", 20, 150),
        "learning_rate":     trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "class_weight":      trial.suggest_categorical("class_weight", ["balanced", None]),
        "objective":         "multiclass",
        "num_class":         4,
        "n_jobs":            -1,
        "random_state":      RANDOM_STATE,
        "verbose":           -1,
    }
    model = LGBMClassifier(**params)
    scores = cross_val_score(
        model, X_train, y_train,
        cv=tscv,
        scoring=scorers[PRIMARY_METRIC],
        n_jobs=1,
    )
    return scores.mean()

In [ ]:
lgbm_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    study_name="lgbm_optuna",
)
lgbm_study.optimize(lgbm_objective, n_trials=N_TRIALS_LGBM, show_progress_bar=True)

print(f"\nMejor trial LGBM:")
print(f"  CV {PRIMARY_METRIC}: {lgbm_study.best_value:.4f}")
print(f"  Params: {lgbm_study.best_params}")

In [ ]:
best_lgbm_params = lgbm_study.best_params.copy()
best_lgbm_params.update({
    "objective":    "multiclass",
    "num_class":    4,
    "n_jobs":       -1,
    "random_state": RANDOM_STATE,
    "verbose":      -1,
})

lgbm_best = LGBMClassifier(**best_lgbm_params)
lgbm_best.fit(X_train, y_train)

lgbm_metrics_val,     y_val_pred_lgbm     = evaluate(lgbm_best, X_val,     y_val,     "val")
lgbm_metrics_holdout, y_holdout_pred_lgbm = evaluate(lgbm_best, X_holdout, y_holdout, "holdout")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (y_true, y_pred, title) in zip(
    axes,
    [(y_val, y_val_pred_lgbm, "VAL"), (y_holdout, y_holdout_pred_lgbm, "HOLDOUT")],
):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d"
    )
    ax.set_title(f"LGBM best — {title}")
plt.tight_layout()
lgbm_cm_path = FIGURES_DIR / "09_lgbm_confusion.png"
plt.savefig(lgbm_cm_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Guardado: {lgbm_cm_path.relative_to(PROJ_ROOT)}")

In [ ]:
feat_imp = pd.Series(
    lgbm_best.feature_importances_,
    index=X_train.columns,
)
top20 = feat_imp.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top20.index[::-1], top20.values[::-1], color="#2E86AB")
ax.set_xlabel("Importancia (splits)")
ax.set_title("LGBM best — Top 20 features")
lgbm_fi_path = FIGURES_DIR / "09_lgbm_feature_importance.png"
plt.savefig(lgbm_fi_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Guardado: {lgbm_fi_path.relative_to(PROJ_ROOT)}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
lgbm_trials_df = lgbm_study.trials_dataframe()
ax.plot(lgbm_trials_df.index, lgbm_trials_df["value"], alpha=0.5, marker=".", color="#2E86AB")
ax.plot(lgbm_trials_df.index, lgbm_trials_df["value"].cummax(), color="#E63946", linewidth=2, label="best so far")
ax.set_xlabel("Trial")
ax.set_ylabel(PRIMARY_METRIC)
ax.set_title("LGBM — historial Optuna")
ax.legend()
ax.grid(alpha=0.3)
lgbm_hist_path = FIGURES_DIR / "09_lgbm_optuna_history.png"
plt.savefig(lgbm_hist_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Guardado: {lgbm_hist_path.relative_to(PROJ_ROOT)}")

In [ ]:
lgbm_model_path = MODELS_DIR / "lgbm_best.joblib"
joblib.dump(lgbm_best, lgbm_model_path)
print(f"Modelo LGBM guardado: {lgbm_model_path.relative_to(PROJ_ROOT)}")

## 6. Curvas de aprendizaje

Diagnóstico bias/variance para XGB y LGBM.

In [ ]:
def plot_learning_curve(estimator, title: str, save_path: Path) -> None:
    train_sizes = np.linspace(0.1, 1.0, 8)
    sizes, train_scores, val_scores = learning_curve(
        estimator=estimator,
        X=X_train,
        y=y_train,
        train_sizes=train_sizes,
        cv=TimeSeriesSplit(n_splits=5),
        scoring=scorers[PRIMARY_METRIC],
        n_jobs=1,
        shuffle=False,
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(sizes, train_scores.mean(axis=1), "o-", label="train", color="#2E86AB")
    ax.fill_between(
        sizes,
        train_scores.mean(axis=1) - train_scores.std(axis=1),
        train_scores.mean(axis=1) + train_scores.std(axis=1),
        alpha=0.15, color="#2E86AB",
    )
    ax.plot(sizes, val_scores.mean(axis=1), "s-", label="val (CV)", color="#E63946")
    ax.fill_between(
        sizes,
        val_scores.mean(axis=1) - val_scores.std(axis=1),
        val_scores.mean(axis=1) + val_scores.std(axis=1),
        alpha=0.15, color="#E63946",
    )
    ax.set_xlabel("Tamaño de train")
    ax.set_ylabel(PRIMARY_METRIC)
    ax.set_title(f"Curva de aprendizaje — {title}")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Guardado: {save_path.relative_to(PROJ_ROOT)}")

In [ ]:
xgb_lc_path = FIGURES_DIR / "09_xgb_learning_curve.png"
plot_learning_curve(xgb_best, "XGB best", xgb_lc_path)

In [ ]:
lgbm_lc_path = FIGURES_DIR / "09_lgbm_learning_curve.png"
plot_learning_curve(lgbm_best, "LGBM best", lgbm_lc_path)

## 7. Registro en MLflow

In [ ]:
def log_model_run(
    run_name: str,
    model_family: str,
    best_params: dict,
    cv_best_score: float,
    metrics_val: dict,
    metrics_holdout: dict,
    model_obj,
    figure_paths: list,
    study: optuna.Study,
) -> None:
    if not MLFLOW_ACTIVE:
        print(f"[MLflow] inactivo — {run_name} no registrado.")
        return

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({
            "model_family": model_family,
            "stage": "advanced_models",
            "validation": "TimeSeriesSplit_5",
            "refit_metric": PRIMARY_METRIC,
        })
        mlflow.log_params(best_params)
        mlflow.log_param("n_features", X_train.shape[1])
        mlflow.log_param("n_train", X_train.shape[0])

        mlflow.log_metric(f"cv_{PRIMARY_METRIC}", cv_best_score)
        for k, v in {**metrics_val, **metrics_holdout}.items():
            mlflow.log_metric(k, float(v))

        for fig_path in figure_paths:
            mlflow.log_artifact(str(fig_path), artifact_path="figures")

        mlflow.sklearn.log_model(model_obj, "model")

        for trial in study.trials:
            if trial.state != optuna.trial.TrialState.COMPLETE:
                continue
            with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
                mlflow.set_tag("model_family", model_family)
                mlflow.set_tag("stage", "optuna_trial")
                mlflow.log_params(trial.params)
                mlflow.log_metric(f"cv_{PRIMARY_METRIC}", trial.value)

        print(f"[MLflow] run registrado: {run.info.run_id}")

In [ ]:
log_model_run(
    run_name=f"xgb_best_depth{xgb_study.best_params['max_depth']}_lr{xgb_study.best_params['learning_rate']:.4f}",
    model_family="xgboost",
    best_params=best_xgb_params,
    cv_best_score=xgb_study.best_value,
    metrics_val=xgb_metrics_val,
    metrics_holdout=xgb_metrics_holdout,
    model_obj=xgb_best,
    figure_paths=[xgb_cm_path, xgb_lc_path, xgb_hist_path],
    study=xgb_study,
)

In [ ]:
log_model_run(
    run_name=f"lgbm_best_leaves{lgbm_study.best_params['num_leaves']}_lr{lgbm_study.best_params['learning_rate']:.4f}",
    model_family="lightgbm",
    best_params=best_lgbm_params,
    cv_best_score=lgbm_study.best_value,
    metrics_val=lgbm_metrics_val,
    metrics_holdout=lgbm_metrics_holdout,
    model_obj=lgbm_best,
    figure_paths=[lgbm_cm_path, lgbm_lc_path, lgbm_hist_path, lgbm_fi_path],
    study=lgbm_study,
)

## 8. Versionado con DVC

Correr en terminal después de ejecutar este notebook:

In [ ]:
print("Para versionar los modelos con DVC (correr en terminal):")
print("  dvc add models/xgb_best.joblib")
print("  dvc add models/lgbm_best.joblib")
print("  dvc push")
print()
print("Después:")
print("  git add models/xgb_best.joblib.dvc models/lgbm_best.joblib.dvc")
print("  git commit -m 'chore(dvc): versiona xgb_best.joblib y lgbm_best.joblib'")